# MAE – Masked Autoencoders for Image Representation Learning
**Kaggle Training Notebook**

Questo notebook è auto-contenuto: scrive tutto il codice sorgente su disco e poi esegue la pipeline completa.

**Dataset richiesto**: `ambityga/imagenet100` (già importato nel notebook)

**Pipeline**:
1. Setup ambiente e dipendenze
2. Scrittura codice sorgente
3. Smoke test (2 epoche, batch ridotto)
4. MAE Pre-training (200 epoche)
5. Supervised ViT Baseline (200 epoche)
6. Linear Probing (90 epoche)
7. Valutazione comparativa

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Percorsi Kaggle ────────────────────────────────────────────────────────────
DATA_ROOT = Path('/kaggle/input/datasets/ambityga/imagenet100')
WORK_DIR  = Path('/kaggle/working')
SRC_DIR   = WORK_DIR / 'src'
EXP_DIR   = WORK_DIR / 'experiments'

sys.path.insert(0, str(WORK_DIR))

# ── W&B API key ────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('W&B API key caricata da Kaggle Secrets.')
except Exception:
    os.environ['WANDB_API_KEY'] = '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'
    print('W&B API key hardcoded.')

# ── Diagnostica GPU + reinstallazione PyTorch compatibile ─────────────────────
# Necessario su Kaggle quando viene assegnata una GPU più recente (L4, A100, H100)
# con compute capability superiore a quella per cui è compilato il PyTorch di default.

nvcc_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
print(nvcc_out.strip() if nvcc_out else 'nvcc non trovato')

# Rileva la versione CUDA dal sistema
cuda_ver_raw = subprocess.run(
    ['nvidia-smi', '--query-gpu=driver_version,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'nvidia-smi: {cuda_ver_raw}')

# Reinstalla PyTorch con CUDA 12.4 (copre compute capability 3.5 → 9.0: T4, V100, A100, L4, H100)
print('\nInstallazione PyTorch cu124 compatibile con tutte le GPU Kaggle...')
!pip install -q wandb tqdm pyyaml
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Compute capability: {torch.cuda.get_device_capability(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # Smoke test veloce per verificare che CUDA funzioni
    x = torch.randn(2, 3, 16, 16).cuda()
    import torch.nn as nn
    conv = nn.Conv2d(3, 8, 3, padding=1).cuda()
    _ = conv(x)
    print('✓ CUDA operativo — nessun errore di kernel.')
else:
    print('⚠ GPU non disponibile: assicurati di aver abilitato il GPU Accelerator nelle impostazioni del notebook.')

## 2. Scrittura codice sorgente

In [ ]:
# Crea struttura directory
for d in [
    SRC_DIR / 'datasets',
    SRC_DIR / 'models',
    SRC_DIR / 'training',
    SRC_DIR / 'evaluation',
    SRC_DIR / 'utils',
    EXP_DIR / 'configs',
    EXP_DIR / 'checkpoints' / 'mae_pretrain',
    EXP_DIR / 'checkpoints' / 'supervised_vit',
    EXP_DIR / 'checkpoints' / 'linear_probe',
    EXP_DIR / 'logs',
]:
    d.mkdir(parents=True, exist_ok=True)

# __init__.py vuoti per i package
for pkg in ['datasets', 'models', 'training', 'evaluation', 'utils']:
    (SRC_DIR / pkg / '__init__.py').touch()
(SRC_DIR / '__init__.py').touch()

print('Directory create.')

In [ ]:
# ── src/utils/misc.py ──────────────────────────────────────────────────────────
(SRC_DIR / 'utils' / 'misc.py').write_text('''
import os, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

def load_dotenv(env_path=None):
    if env_path is None:
        current = Path.cwd()
        for parent in [current, *current.parents]:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break
    if env_path is None or not Path(env_path).exists():
        return
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            key = key.strip(); value = value.strip()
            if key and key not in os.environ:
                os.environ[key] = value

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def get_device(use_cuda=True):
    if use_cuda and torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"[Device] GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("[Device] CPU")
    return device

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def format_eta(seconds):
    h = int(seconds // 3600); m = int((seconds % 3600) // 60); s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"
''')

# ── src/utils/config.py ────────────────────────────────────────────────────────
(SRC_DIR / 'utils' / 'config.py').write_text('''
import yaml
from pathlib import Path

def load_config(config_path):
    with open(config_path) as f:
        return yaml.safe_load(f)

def get_nested(cfg, *keys, default=None):
    d = cfg
    for k in keys:
        if not isinstance(d, dict) or k not in d: return default
        d = d[k]
    return d
''')

print('utils scritti.')

In [ ]:
# ── src/utils/logger.py ────────────────────────────────────────────────────────
(SRC_DIR / 'utils' / 'logger.py').write_text('''
from pathlib import Path
import torch

class UnifiedLogger:
    def __init__(self, log_dir, experiment_name, config, use_wandb=True, use_tensorboard=True):
        self.experiment_name = experiment_name
        self.use_wandb = use_wandb
        self.use_tensorboard = use_tensorboard
        tb_dir = Path(log_dir) / experiment_name
        if self.use_tensorboard:
            from torch.utils.tensorboard import SummaryWriter
            tb_dir.mkdir(parents=True, exist_ok=True)
            self._writer = SummaryWriter(log_dir=str(tb_dir))
        else:
            self._writer = None
        if self.use_wandb:
            import wandb
            wandb_project = config.get("logging", {}).get("wandb_project", "mae-project")
            wandb.init(project=wandb_project, name=experiment_name, config=config,
                       dir=str(tb_dir), resume="allow")
            self._wandb = wandb
        else:
            self._wandb = None

    def log_scalar(self, tag, value, step):
        if self._writer: self._writer.add_scalar(tag, value, global_step=step)
        if self._wandb: self._wandb.log({tag: value}, step=step)

    def log_scalars(self, tag_value_dict, step):
        for tag, value in tag_value_dict.items():
            if self._writer: self._writer.add_scalar(tag, value, global_step=step)
        if self._wandb: self._wandb.log(tag_value_dict, step=step)

    def log_image(self, tag, image, step):
        if self._writer:
            if image.dim() == 3: self._writer.add_image(tag, image, global_step=step)
            else: self._writer.add_images(tag, image, global_step=step)
        if self._wandb:
            import wandb
            if image.dim() == 3:
                self._wandb.log({tag: wandb.Image(image.permute(1,2,0).cpu().numpy())}, step=step)
            else:
                self._wandb.log({tag: [wandb.Image(i.permute(1,2,0).cpu().numpy()) for i in image]}, step=step)

    def finish(self):
        if self._writer: self._writer.close()
        if self._wandb: self._wandb.finish()
''')

# ── src/utils/checkpoint.py ────────────────────────────────────────────────────
(SRC_DIR / 'utils' / 'checkpoint.py').write_text('''
from pathlib import Path
import torch
import torch.nn as nn

def save_checkpoint(state, checkpoint_dir, filename):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    path = checkpoint_dir / filename
    torch.save(state, path)
    print(f"[Checkpoint] Saved → {path}")

def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None, device=None):
    if device is None: device = torch.device("cpu")
    ckpt = torch.load(checkpoint_path, map_location=device)
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if missing: print(f"[Checkpoint] Missing: {missing}")
    if unexpected: print(f"[Checkpoint] Unexpected: {unexpected}")
    if optimizer and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler and "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    print(f"[Checkpoint] Loaded from {checkpoint_path} (epoch {ckpt.get('epoch','?')})")
    return ckpt

def load_encoder_weights(mae_checkpoint_path, vit_classifier, device=None):
    if device is None: device = torch.device("cpu")
    ckpt = torch.load(mae_checkpoint_path, map_location=device)
    mae_state = ckpt["model_state_dict"]
    encoder_state = {k[len("encoder."):]: v for k, v in mae_state.items() if k.startswith("encoder.")}
    missing, unexpected = vit_classifier.encoder.load_state_dict(encoder_state, strict=False)
    if missing: print(f"[Checkpoint] Encoder missing: {missing}")
    if unexpected: print(f"[Checkpoint] Encoder unexpected: {unexpected}")
    print(f"[Checkpoint] Encoder weights loaded from {mae_checkpoint_path}")
''')

# ── src/utils/__init__.py ──────────────────────────────────────────────────────
(SRC_DIR / 'utils' / '__init__.py').write_text('''
from .checkpoint import load_checkpoint, load_encoder_weights, save_checkpoint
from .config import load_config
from .logger import UnifiedLogger
from .misc import count_parameters, format_eta, get_device, set_seed
''')

print('utils checkpoint/logger/__init__ scritti.')

In [ ]:
# ── src/datasets/imagenet100.py ────────────────────────────────────────────────
(SRC_DIR / 'datasets' / 'imagenet100.py').write_text('''
from pathlib import Path
import torch
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import datasets, transforms

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_mae_transform(image_size=224):
    return transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.2, 1.0),
                                     interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def build_supervised_transform(image_size=224, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(image_size,
                                         interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

def _find_split_dirs(root, split):
    root = Path(root)
    direct = root / split
    if direct.is_dir():
        return [direct]
    candidates = sorted(
        d for d in root.iterdir()
        if d.is_dir() and (
            d.name == f"{split}.X" or
            (d.name.startswith(f"{split}.X") and d.name[len(split)+2:].isdigit())
        )
    )
    if candidates:
        return candidates
    available = [d.name for d in root.iterdir() if d.is_dir()]
    raise FileNotFoundError(f"Cannot find split {split!r} in {root}. Available: {available}")

def _build_global_class_to_idx(split_dirs):
    all_classes = set()
    for d in split_dirs:
        all_classes.update(e.name for e in d.iterdir() if e.is_dir())
    return {cls: idx for idx, cls in enumerate(sorted(all_classes))}

class _FixedClassImageFolder(datasets.ImageFolder):
    def __init__(self, root, class_to_idx_override, **kwargs):
        self._class_to_idx_override = class_to_idx_override
        super().__init__(root, **kwargs)
    def find_classes(self, directory):
        classes = sorted(
            e.name for e in Path(directory).iterdir()
            if e.is_dir() and e.name in self._class_to_idx_override
        )
        return classes, {c: self._class_to_idx_override[c] for c in classes}

class ImageNet100Dataset(torch.utils.data.Dataset):
    def __init__(self, root, split, transform=None):
        root = Path(root)
        split_dirs = _find_split_dirs(root, split)
        class_to_idx = _build_global_class_to_idx(split_dirs)
        if len(split_dirs) == 1:
            self._dataset = _FixedClassImageFolder(
                str(split_dirs[0]), class_to_idx_override=class_to_idx, transform=transform)
        else:
            subs = [_FixedClassImageFolder(str(d), class_to_idx_override=class_to_idx,
                                            transform=transform) for d in split_dirs]
            self._dataset = ConcatDataset(subs)
        self._class_to_idx = class_to_idx
        self._num_classes  = len(class_to_idx)
        print(f"[Dataset] {split!r} → {[d.name for d in split_dirs]} | "
              f"{len(self)} images, {self._num_classes} classes")
    def __len__(self): return len(self._dataset)
    def __getitem__(self, idx): return self._dataset[idx]
    @property
    def num_classes(self): return self._num_classes
    @property
    def class_to_idx(self): return self._class_to_idx

def build_dataloader(root, split, transform, batch_size, num_workers,
                     pin_memory=True, drop_last=False, shuffle=None):
    dataset = ImageNet100Dataset(root=root, split=split, transform=transform)
    if shuffle is None: shuffle = (split == "train")
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                        num_workers=num_workers, pin_memory=pin_memory, drop_last=drop_last)
    print(f"[DataLoader] {split}: {len(loader)} batches (bs={batch_size})")
    return loader
''')

print('datasets scritto.')

In [ ]:
# ── src/models/patch_embed.py ──────────────────────────────────────────────────
(SRC_DIR / 'models' / 'patch_embed.py').write_text('''
import torch, torch.nn as nn

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size; self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self._num_patches = self.grid_size ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    @property
    def num_patches(self): return self._num_patches
    def forward(self, x):
        x = self.proj(x)           # (B, D, H/P, W/P)
        x = x.flatten(2)           # (B, D, N)
        x = x.transpose(1, 2)      # (B, N, D)
        return x
''')

# ── src/models/vit_encoder.py ──────────────────────────────────────────────────
(SRC_DIR / 'models' / 'vit_encoder.py').write_text('''
import numpy as np, torch, torch.nn as nn
from .patch_embed import PatchEmbed

def get_2d_sincos_pos_embed(embed_dim, grid_size):
    half_dim = embed_dim // 2
    omega = np.arange(half_dim // 2, dtype=np.float64)
    omega /= half_dim // 2
    omega = 1.0 / (10000 ** omega)
    grid_h = np.arange(grid_size, dtype=np.float64)
    grid_w = np.arange(grid_size, dtype=np.float64)
    grid_h, grid_w = np.meshgrid(grid_h, grid_w, indexing="ij")
    grid_h = grid_h.reshape(-1); grid_w = grid_w.reshape(-1)
    out_h = np.outer(grid_h, omega); out_w = np.outer(grid_w, omega)
    emb = np.concatenate([
        np.sin(out_h), np.cos(out_h), np.sin(out_w), np.cos(out_w)
    ], axis=1)
    return emb.astype(np.float32)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        n = self.norm1(x); a, _ = self.attn(n, n, n, need_weights=False)
        x = x + a; x = x + self.mlp(self.norm2(x))
        return x

class ViTEncoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        pos = get_2d_sincos_pos_embed(embed_dim, self.patch_embed.grid_size)
        pos_with_cls = np.zeros((1, num_patches + 1, embed_dim), dtype=np.float32)
        pos_with_cls[0, 1:] = pos
        self.register_buffer("pos_embed", torch.from_numpy(pos_with_cls))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
                                     for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.norm(x)
''')

print('models patch_embed + vit_encoder scritti.')

In [ ]:
# ── src/models/mae_decoder.py ──────────────────────────────────────────────────
(SRC_DIR / 'models' / 'mae_decoder.py').write_text('''
import numpy as np, torch, torch.nn as nn
from .vit_encoder import TransformerBlock, get_2d_sincos_pos_embed

class MAEDecoder(nn.Module):
    def __init__(self, num_patches=196, grid_size=14, encoder_embed_dim=768,
                 decoder_embed_dim=512, decoder_depth=8, decoder_num_heads=16,
                 patch_size=16, in_channels=3, mlp_ratio=4.0):
        super().__init__()
        self.num_patches = num_patches
        self.patch_dim   = patch_size * patch_size * in_channels
        self.embed      = nn.Linear(encoder_embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        pos = get_2d_sincos_pos_embed(decoder_embed_dim, grid_size)
        self.register_buffer("pos_embed", torch.from_numpy(pos).unsqueeze(0))
        self.blocks = nn.ModuleList([TransformerBlock(decoder_embed_dim, decoder_num_heads, mlp_ratio)
                                     for _ in range(decoder_depth)])
        self.norm = nn.LayerNorm(decoder_embed_dim)
        self.head = nn.Linear(decoder_embed_dim, self.patch_dim, bias=True)
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, visible_tokens, visible_indices, mask_indices):
        B = visible_tokens.shape[0]
        N = self.num_patches
        vis = self.embed(visible_tokens)
        pe  = self.pos_embed.expand(B, -1, -1)
        vis = vis + torch.gather(pe, 1, visible_indices.unsqueeze(-1).expand(-1, -1, pe.shape[-1]))
        msk = self.mask_token.expand(B, mask_indices.shape[1], -1)
        msk = msk + torch.gather(pe, 1, mask_indices.unsqueeze(-1).expand(-1, -1, pe.shape[-1]))
        D = vis.shape[-1]
        full = torch.zeros(B, N, D, device=vis.device, dtype=vis.dtype)
        full.scatter_(1, visible_indices.unsqueeze(-1).expand(-1, -1, D), vis)
        full.scatter_(1, mask_indices.unsqueeze(-1).expand(-1, -1, D), msk)
        for blk in self.blocks: full = blk(full)
        return self.head(self.norm(full))
''')

# ── src/models/mae.py ──────────────────────────────────────────────────────────
(SRC_DIR / 'models' / 'mae.py').write_text('''
import torch, torch.nn as nn
from .mae_decoder import MAEDecoder
from .vit_encoder import ViTEncoder

class MAE(nn.Module):
    def __init__(self, encoder, decoder, mask_ratio=0.75, norm_pix_loss=True):
        super().__init__()
        self.encoder = encoder; self.decoder = decoder
        self.mask_ratio = mask_ratio; self.norm_pix_loss = norm_pix_loss
        self.patch_size  = encoder.patch_embed.patch_size
        self.num_patches = encoder.patch_embed.num_patches

    def patchify(self, imgs):
        p = self.patch_size; B, C, H, W = imgs.shape
        h, w = H // p, W // p
        x = imgs.reshape(B, C, h, p, w, p).permute(0, 2, 4, 3, 5, 1)
        return x.reshape(B, h * w, p * p * C)

    def unpatchify(self, patches):
        p = self.patch_size; B, N, _ = patches.shape
        h = w = int(N ** 0.5); C = 3
        x = patches.reshape(B, h, w, p, p, C).permute(0, 5, 1, 3, 2, 4)
        return x.reshape(B, C, h * p, w * p)

    def random_masking(self, x, mask_ratio):
        B, N, D = x.shape
        num_keep = int(N * (1 - mask_ratio))
        noise = torch.rand(B, N, device=x.device)
        ids = torch.argsort(noise, dim=1)
        vis_idx  = ids[:, :num_keep]
        mask_idx = ids[:, num_keep:]
        vis_tok  = torch.gather(x, 1, vis_idx.unsqueeze(-1).expand(-1, -1, D))
        mask = torch.ones(B, N, device=x.device)
        mask.scatter_(1, vis_idx, 0.0)
        return vis_tok, mask, vis_idx, mask_idx

    def forward(self, imgs):
        B = imgs.shape[0]
        target = self.patchify(imgs)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var  = target.var(dim=-1, keepdim=True, unbiased=False)
            target = (target - mean) / (var + 1e-6).sqrt()
        x = self.encoder.patch_embed(imgs) + self.encoder.pos_embed[:, 1:, :]
        vis_tok, mask, vis_idx, mask_idx = self.random_masking(x, self.mask_ratio)
        cls = self.encoder.cls_token.expand(B, -1, -1)
        vis_with_cls = torch.cat([cls, vis_tok], dim=1)
        cls_pe = self.encoder.pos_embed[:, :1, :].expand(B, -1, -1)
        vis_with_cls = vis_with_cls + torch.cat([cls_pe, torch.zeros_like(vis_tok)], dim=1)
        for blk in self.encoder.blocks: vis_with_cls = blk(vis_with_cls)
        vis_with_cls = self.encoder.norm(vis_with_cls)
        encoded_vis = vis_with_cls[:, 1:, :]
        pred = self.decoder(encoded_vis, vis_idx, mask_idx)
        loss = ((pred - target) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        return loss, pred, mask

def build_mae(cfg):
    ec = cfg["model"]["encoder"]; dc = cfg["model"]["decoder"]
    img_size   = cfg["model"].get("img_size", 224)
    patch_size = cfg["model"].get("patch_size", 16)
    grid_size  = img_size // patch_size
    enc = ViTEncoder(img_size=img_size, patch_size=patch_size, in_channels=3,
                     embed_dim=ec["embed_dim"], depth=ec["depth"], num_heads=ec["num_heads"],
                     mlp_ratio=ec.get("mlp_ratio", 4.0), dropout=ec.get("dropout", 0.0))
    dec = MAEDecoder(num_patches=grid_size**2, grid_size=grid_size,
                     encoder_embed_dim=ec["embed_dim"],
                     decoder_embed_dim=dc["embed_dim"], decoder_depth=dc["depth"],
                     decoder_num_heads=dc["num_heads"], patch_size=patch_size,
                     mlp_ratio=dc.get("mlp_ratio", 4.0))
    return MAE(enc, dec, mask_ratio=cfg["model"].get("mask_ratio", 0.75),
               norm_pix_loss=cfg["model"].get("norm_pix_loss", True))
''')

# ── src/models/vit_classifier.py ───────────────────────────────────────────────
(SRC_DIR / 'models' / 'vit_classifier.py').write_text('''
import torch, torch.nn as nn
from .vit_encoder import ViTEncoder

class ViTClassifier(nn.Module):
    def __init__(self, encoder, num_classes=100, dropout=0.0):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(encoder.patch_embed.proj.out_channels, num_classes)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
    def freeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad = True
    def forward(self, x):
        features = self.encoder(x)
        return self.head(self.dropout(features[:, 0]))

def build_vit_classifier(cfg, pretrained_encoder_path=None, device=None):
    ec = cfg["model"]["encoder"]
    enc = ViTEncoder(img_size=cfg["model"].get("img_size", 224),
                     patch_size=cfg["model"].get("patch_size", 16),
                     in_channels=3, embed_dim=ec["embed_dim"], depth=ec["depth"],
                     num_heads=ec["num_heads"], mlp_ratio=ec.get("mlp_ratio", 4.0),
                     dropout=ec.get("dropout", 0.0))
    model = ViTClassifier(enc, num_classes=cfg["model"].get("num_classes", 100),
                           dropout=cfg["model"].get("head_dropout", 0.0))
    if pretrained_encoder_path is not None:
        from src.utils.checkpoint import load_encoder_weights
        load_encoder_weights(pretrained_encoder_path, model, device=device)
        model.freeze_encoder()
        print("[Model] Encoder frozen for linear probing.")
    return model
''')

# ── src/models/__init__.py ─────────────────────────────────────────────────────
(SRC_DIR / 'models' / '__init__.py').write_text('''
from .mae import MAE, build_mae
from .vit_classifier import ViTClassifier, build_vit_classifier
''')

print('models scritti.')

In [ ]:
# ── src/training/ ──────────────────────────────────────────────────────────────
(SRC_DIR / 'training' / 'losses.py').write_text('''
import torch.nn.functional as F

def mae_reconstruction_loss(pred, target, mask):
    loss = ((pred - target) ** 2).mean(dim=-1)
    return (loss * mask).sum() / mask.sum()

def cross_entropy_loss(logits, labels, label_smoothing=0.0):
    return F.cross_entropy(logits, labels, label_smoothing=label_smoothing)
''')

(SRC_DIR / 'training' / 'optimizer.py').write_text('''
import math
import torch, torch.nn as nn

def build_optimizer(model, optimizer_cfg):
    no_decay = ["bias", "norm"]
    decay    = [p for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)]
    no_decay_ = [p for n, p in model.named_parameters()
                 if p.requires_grad and any(nd in n for nd in no_decay)]
    param_groups = [
        {"params": decay,    "weight_decay": optimizer_cfg.get("weight_decay", 0.05)},
        {"params": no_decay_,"weight_decay": 0.0},
    ]
    name = optimizer_cfg.get("name", "adamw").lower()
    lr   = optimizer_cfg["lr"]
    if name == "adamw":
        betas = tuple(optimizer_cfg.get("betas", [0.9, 0.999]))
        return torch.optim.AdamW(param_groups, lr=lr, betas=betas)
    elif name == "sgd":
        return torch.optim.SGD(param_groups, lr=lr,
                               momentum=optimizer_cfg.get("momentum", 0.9), nesterov=True)
    raise ValueError(f"Unsupported optimizer: {name}")

def build_scheduler(optimizer, scheduler_cfg, num_epochs):
    warmup = scheduler_cfg.get("warmup_epochs", 0)
    min_lr = scheduler_cfg.get("min_lr", 0.0)
    base_lr = optimizer.param_groups[0]["lr"]
    name = scheduler_cfg.get("name", "cosine").lower()
    if name == "cosine":
        def lr_lambda(epoch):
            if epoch < warmup: return (epoch + 1) / max(warmup, 1)
            progress = (epoch - warmup) / max(num_epochs - warmup, 1)
            cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
            return max(min_lr / base_lr if base_lr > 0 else 0.0, cosine)
    else:
        raise ValueError(f"Unsupported scheduler: {name}")
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
''')

print('training losses + optimizer scritti.')

In [ ]:
# ── Trainer condiviso (base class con logica comune) ───────────────────────────
_trainer_base = '''
import time
from pathlib import Path
import torch
from tqdm.auto import tqdm
from src.training.losses import cross_entropy_loss
from src.utils.checkpoint import save_checkpoint
from src.utils.misc import format_eta

class _BaseClassificationTrainer:
    def __init__(self, model, optimizer, scheduler, train_loader, val_loader,
                 logger, cfg, device, checkpoint_dir):
        self.model = model; self.optimizer = optimizer; self.scheduler = scheduler
        self.train_loader = train_loader; self.val_loader = val_loader
        self.logger = logger; self.cfg = cfg; self.device = device
        self.checkpoint_dir = Path(checkpoint_dir)
        self.epochs    = cfg["training"]["epochs"]
        self.log_freq  = cfg["logging"].get("log_freq", 50)
        self.save_every= cfg["checkpointing"].get("save_every", 10)
        self.grad_clip = cfg["training"].get("grad_clip", None)
        self.label_smoothing = cfg["model"].get("label_smoothing", 0.0)
        self.global_step = 0

    def train_one_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0; total_correct = 0; total_samples = 0
        pbar = tqdm(self.train_loader,
                    desc=f"Epoch {epoch+1}/{self.epochs} [train]",
                    leave=True, dynamic_ncols=True)
        for step, (imgs, labels) in enumerate(pbar):
            imgs   = imgs.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            logits = self.model(imgs)
            loss   = cross_entropy_loss(logits, labels, self.label_smoothing)
            self.optimizer.zero_grad(); loss.backward()
            if self.grad_clip: torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_samples += labels.size(0); total_loss += loss.item()
            self.global_step += 1
            lr = self.optimizer.param_groups[0]["lr"]
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "top1": f"{100.*total_correct/total_samples:.1f}%",
                "lr":   f"{lr:.2e}",
            })
            if (step + 1) % self.log_freq == 0:
                self.logger.log_scalars({"train/loss_step": loss.item(), "train/lr": lr},
                                        step=self.global_step)
        return {"loss": total_loss/len(self.train_loader),
                "top1_acc": 100.*total_correct/total_samples,
                "lr": self.optimizer.param_groups[0]["lr"]}

    @torch.no_grad()
    def evaluate(self, epoch):
        self.model.eval()
        total_loss = 0.0; top1 = 0; top5 = 0; n = 0
        pbar = tqdm(self.val_loader,
                    desc=f"Epoch {epoch+1}/{self.epochs} [val]  ",
                    leave=False, dynamic_ncols=True)
        for imgs, labels in pbar:
            imgs = imgs.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            logits = self.model(imgs)
            total_loss += cross_entropy_loss(logits, labels).item()
            _, p5 = logits.topk(5, dim=1)
            top1 += (p5[:,0] == labels).sum().item()
            top5 += (p5 == labels.unsqueeze(1)).any(1).sum().item()
            n += labels.size(0)
            pbar.set_postfix({"top1": f"{100.*top1/n:.1f}%", "top5": f"{100.*top5/n:.1f}%"})
        return {"val_loss": total_loss/len(self.val_loader),
                "val_top1": 100.*top1/n, "val_top5": 100.*top5/n}

    def save_checkpoint(self, epoch, metrics, is_best=False, suffix=""):
        state = {"epoch": epoch, "model_state_dict": self.model.state_dict(),
                 "optimizer_state_dict": self.optimizer.state_dict(),
                 "scheduler_state_dict": self.scheduler.state_dict(),
                 "metrics": metrics, "config": self.cfg}
        fname = f"checkpoint_epoch_{epoch+1:04d}{suffix}.pth"
        save_checkpoint(state, self.checkpoint_dir, fname)
        if is_best: save_checkpoint(state, self.checkpoint_dir, "checkpoint_best.pth")
'''

(SRC_DIR / 'training' / 'trainer_supervised.py').write_text(_trainer_base + '''
class SupervisedTrainer(_BaseClassificationTrainer):
    def train(self, start_epoch=0):
        print(f"\\n{'='*55}\\n  Supervised ViT: {self.epochs} epochs\\n{'='*55}")
        best_top1 = 0.0
        for epoch in range(start_epoch, self.epochs):
            tm = self.train_one_epoch(epoch)
            vm = self.evaluate(epoch)
            self.scheduler.step()
            print(f"  → loss={tm[\'loss\']:.4f}  train_top1={tm[\'top1_acc\']:.2f}%  "
                  f"val_top1={vm[\'val_top1\']:.2f}%  val_top5={vm[\'val_top5\']:.2f}%")
            self.logger.log_scalars({"train/loss": tm["loss"], "train/top1": tm["top1_acc"],
                "train/lr": tm["lr"], "val/loss": vm["val_loss"],
                "val/top1": vm["val_top1"], "val/top5": vm["val_top5"]}, step=epoch+1)
            is_best = vm["val_top1"] > best_top1
            if is_best: best_top1 = vm["val_top1"]
            if (epoch+1) % self.save_every == 0 or is_best:
                self.save_checkpoint(epoch, vm, is_best=is_best)
        self.save_checkpoint(self.epochs-1, {}, suffix="_final")
        print(f"Best Val Top-1: {best_top1:.2f}%")
''')

(SRC_DIR / 'training' / 'trainer_linear_probe.py').write_text(_trainer_base + '''
class LinearProbeTrainer(_BaseClassificationTrainer):
    def train_one_epoch(self, epoch):
        self.model.encoder.eval()  # keep frozen encoder in eval mode
        return super().train_one_epoch(epoch)
    def train(self, start_epoch=0):
        print(f"\\n{'='*55}\\n  Linear Probe: {self.epochs} epochs\\n{'='*55}")
        best_top1 = 0.0
        for epoch in range(start_epoch, self.epochs):
            tm = self.train_one_epoch(epoch)
            vm = self.evaluate(epoch)
            self.scheduler.step()
            print(f"  → loss={tm[\'loss\']:.4f}  train_top1={tm[\'top1_acc\']:.2f}%  "
                  f"val_top1={vm[\'val_top1\']:.2f}%  val_top5={vm[\'val_top5\']:.2f}%")
            self.logger.log_scalars({"train/loss": tm["loss"], "train/top1": tm["top1_acc"],
                "train/lr": tm["lr"], "val/loss": vm["val_loss"],
                "val/top1": vm["val_top1"], "val/top5": vm["val_top5"]}, step=epoch+1)
            is_best = vm["val_top1"] > best_top1
            if is_best: best_top1 = vm["val_top1"]
            if (epoch+1) % self.save_every == 0 or is_best:
                self.save_checkpoint(epoch, vm, is_best=is_best)
        self.save_checkpoint(self.epochs-1, {}, suffix="_final")
        print(f"Best Val Top-1: {best_top1:.2f}%")
''')

print('trainers con tqdm scritti.')

In [ ]:
# ── src/training/trainer_mae.py ────────────────────────────────────────────────
(SRC_DIR / 'training' / 'trainer_mae.py').write_text('''
import time
from pathlib import Path
import torch
from tqdm.auto import tqdm
from src.utils.checkpoint import save_checkpoint
from src.utils.misc import format_eta

class MAETrainer:
    def __init__(self, model, optimizer, scheduler, train_loader,
                 logger, cfg, device, checkpoint_dir):
        self.model = model; self.optimizer = optimizer; self.scheduler = scheduler
        self.train_loader = train_loader; self.logger = logger; self.cfg = cfg
        self.device = device; self.checkpoint_dir = Path(checkpoint_dir)
        self.epochs     = cfg["training"]["epochs"]
        self.log_freq   = cfg["logging"].get("log_freq", 50)
        self.save_every = cfg["checkpointing"].get("save_every", 20)
        self.grad_clip  = cfg["training"].get("grad_clip", None)
        self.global_step = 0

    def train_one_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0
        pbar = tqdm(self.train_loader,
                    desc=f"Epoch {epoch+1}/{self.epochs} [MAE]",
                    leave=True, dynamic_ncols=True)
        for step, (imgs, _) in enumerate(pbar):
            imgs = imgs.to(self.device, non_blocking=True)
            loss, _, _ = self.model(imgs)
            self.optimizer.zero_grad(); loss.backward()
            if self.grad_clip: torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
            total_loss += loss.item(); self.global_step += 1
            lr = self.optimizer.param_groups[0]["lr"]
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{lr:.2e}"})
            if (step + 1) % self.log_freq == 0:
                self.logger.log_scalars({"train/loss_step": loss.item(), "train/lr": lr},
                                        step=self.global_step)
        return {"loss": total_loss/len(self.train_loader), "lr": self.optimizer.param_groups[0]["lr"]}

    def train(self, start_epoch=0):
        print(f"\\n{\'=\'*55}\\n  MAE Pre-training: {self.epochs} epochs\\n{\'=\'*55}")
        best_loss = float("inf")
        for epoch in range(start_epoch, self.epochs):
            m = self.train_one_epoch(epoch)
            self.scheduler.step()
            print(f"  → avg_loss={m[\'loss\']:.4f}  lr={m[\'lr\']:.2e}")
            self.logger.log_scalars({"train/loss_epoch": m["loss"], "train/lr_epoch": m["lr"]},
                                    step=epoch+1)
            is_best = m["loss"] < best_loss
            if is_best: best_loss = m["loss"]
            if (epoch+1) % self.save_every == 0 or is_best:
                self.save_checkpoint(epoch, is_best=is_best)
        self.save_checkpoint(self.epochs-1, suffix="_final")
        print("MAE pre-training complete.")

    def save_checkpoint(self, epoch, is_best=False, suffix=""):
        state = {"epoch": epoch, "model_state_dict": self.model.state_dict(),
                 "optimizer_state_dict": self.optimizer.state_dict(),
                 "scheduler_state_dict": self.scheduler.state_dict(),
                 "config": self.cfg}
        fname = f"checkpoint_epoch_{epoch+1:04d}{suffix}.pth"
        save_checkpoint(state, self.checkpoint_dir, fname)
        if is_best: save_checkpoint(state, self.checkpoint_dir, "checkpoint_best.pth")
''')

# ── src/evaluation/ ────────────────────────────────────────────────────────────
(SRC_DIR / 'evaluation' / 'metrics.py').write_text('''
import torch

def top_k_accuracy(logits, labels, k=1):
    with torch.no_grad():
        _, topk = logits.topk(k, dim=1)
        return 100.0 * (topk == labels.unsqueeze(1)).any(1).sum().item() / labels.size(0)

class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self._sum = 0.0; self._count = 0
    def update(self, val, n=1): self._sum += val * n; self._count += n
    @property
    def avg(self): return self._sum / self._count if self._count > 0 else 0.0
    @property
    def count(self): return self._count
''')

(SRC_DIR / 'evaluation' / 'evaluator.py').write_text('''
import torch, torch.nn as nn
from .metrics import AverageMeter

class Evaluator:
    def __init__(self, model, val_loader, device, num_classes=100):
        self.model = model; self.val_loader = val_loader
        self.device = device; self.num_classes = num_classes
    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        loss_m = AverageMeter(); top1_m = AverageMeter(); top5_m = AverageMeter()
        crit = nn.CrossEntropyLoss()
        for imgs, labels in self.val_loader:
            imgs = imgs.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            B = labels.size(0)
            logits = self.model(imgs)
            loss_m.update(crit(logits, labels).item(), B)
            _, p5 = logits.topk(min(5, self.num_classes), dim=1)
            top1_m.update(100.*(p5[:,0]==labels).sum().item()/B, B)
            top5_m.update(100.*(p5==labels.unsqueeze(1)).any(1).sum().item()/B, B)
        return {"top1": top1_m.avg, "top5": top5_m.avg,
                "loss": loss_m.avg, "num_samples": top1_m.count}
    def print_report(self, results, label="Model"):
        w = 60
        print("="*w)
        print(f"  {label}")
        print("-"*w)
        print(f"  Top-1 Accuracy: {results[\'top1\']:>8.2f}%")
        print(f"  Top-5 Accuracy: {results[\'top5\']:>8.2f}%")
        print(f"  Val Loss:       {results[\'loss\']:>8.4f}")
        print(f"  Num Samples:    {results[\'num_samples\']:>8}")
        print("="*w)

def print_comparison(results_list):
    w = 64
    print("\\n" + "="*w)
    print("  ImageNet100 Evaluation Results")
    print("="*w)
    print(f"  {\'Model\':<36} {\'Top-1 (%)\':>8}  {\'Top-5 (%)\':>8}")
    print("-"*w)
    for label, r in results_list:
        print(f"  {label:<36} {r[\'top1\']:>8.2f}  {r[\'top5\']:>8.2f}")
    print("="*w + "\\n")
''')

print('MAETrainer con tqdm + evaluation scritti.')

In [ ]:
# ── Config YAML per Kaggle ─────────────────────────────────────────────────────
import yaml

# Batch size ridotto per P100/T4 (16 GB VRAM)
# Per GPU più grandi (A100 40 GB) puoi aumentare a 256
BATCH_SIZE = 128

mae_cfg = {
    "experiment": {"name": "mae_pretrain", "seed": 42},
    "data": {"root": str(DATA_ROOT), "image_size": 224,
             "batch_size": BATCH_SIZE, "num_workers": 4, "pin_memory": True},
    "model": {
        "patch_size": 16, "img_size": 224, "mask_ratio": 0.75, "norm_pix_loss": True,
        "encoder": {"embed_dim": 768, "depth": 12, "num_heads": 12, "mlp_ratio": 4.0, "dropout": 0.0},
        "decoder": {"embed_dim": 512, "depth": 8, "num_heads": 16, "mlp_ratio": 4.0},
    },
    "training": {
        "epochs": 200, "warmup_epochs": 40, "grad_clip": 1.0,
        "optimizer": {"name": "adamw", "lr": 1.5e-4, "weight_decay": 0.05, "betas": [0.9, 0.95]},
        "scheduler": {"name": "cosine", "warmup_epochs": 40, "min_lr": 0.0},
    },
    "logging": {"log_dir": str(EXP_DIR / "logs"), "log_freq": 50,
                "use_wandb": True, "use_tensorboard": False, "wandb_project": "mae-imagenet100"},
    "checkpointing": {"checkpoint_dir": str(EXP_DIR / "checkpoints" / "mae_pretrain"),
                      "save_every": 20, "resume": None},
}

sup_cfg = {
    "experiment": {"name": "supervised_vit_baseline", "seed": 42},
    "data": {"root": str(DATA_ROOT), "image_size": 224,
             "batch_size": BATCH_SIZE, "num_workers": 4, "pin_memory": True},
    "model": {
        "patch_size": 16, "img_size": 224, "num_classes": 100,
        "label_smoothing": 0.1, "head_dropout": 0.0,
        "encoder": {"embed_dim": 768, "depth": 12, "num_heads": 12, "mlp_ratio": 4.0, "dropout": 0.1},
    },
    "training": {
        "epochs": 200, "warmup_epochs": 20, "grad_clip": 1.0,
        "optimizer": {"name": "adamw", "lr": 1e-3, "weight_decay": 0.05, "betas": [0.9, 0.999]},
        "scheduler": {"name": "cosine", "warmup_epochs": 20, "min_lr": 1e-6},
    },
    "logging": {"log_dir": str(EXP_DIR / "logs"), "log_freq": 50,
                "use_wandb": True, "use_tensorboard": False, "wandb_project": "mae-imagenet100"},
    "checkpointing": {"checkpoint_dir": str(EXP_DIR / "checkpoints" / "supervised_vit"),
                      "save_every": 10, "resume": None},
}

probe_cfg = {
    "experiment": {"name": "linear_probe_mae", "seed": 42},
    "data": {"root": str(DATA_ROOT), "image_size": 224,
             "batch_size": BATCH_SIZE, "num_workers": 4, "pin_memory": True},
    "model": {
        "patch_size": 16, "img_size": 224, "num_classes": 100, "head_dropout": 0.0,
        "encoder": {"embed_dim": 768, "depth": 12, "num_heads": 12, "mlp_ratio": 4.0, "dropout": 0.0},
        "pretrained_encoder": str(EXP_DIR / "checkpoints" / "mae_pretrain" / "checkpoint_best.pth"),
        "freeze_encoder": True,
    },
    "training": {
        "epochs": 90, "warmup_epochs": 10, "grad_clip": None,
        "optimizer": {"name": "adamw", "lr": 1e-3, "weight_decay": 0.0, "betas": [0.9, 0.999]},
        "scheduler": {"name": "cosine", "warmup_epochs": 10, "min_lr": 0.0},
    },
    "logging": {"log_dir": str(EXP_DIR / "logs"), "log_freq": 50,
                "use_wandb": True, "use_tensorboard": False, "wandb_project": "mae-imagenet100"},
    "checkpointing": {"checkpoint_dir": str(EXP_DIR / "checkpoints" / "linear_probe"),
                      "save_every": 10, "resume": None},
}

for name, cfg in [("mae_pretrain", mae_cfg), ("supervised_vit", sup_cfg), ("linear_probe", probe_cfg)]:
    p = EXP_DIR / 'configs' / f'{name}.yaml'
    p.write_text(yaml.dump(cfg, default_flow_style=False))

print('Config YAML scritti in', EXP_DIR / 'configs')

## 3. Smoke Test (2 epoche, batch ridotto)
Verifica che tutto il codice funzioni prima di lanciare il training completo.

In [ ]:
import copy, torch
from src.utils.misc import set_seed, get_device, count_parameters
from src.utils.logger import UnifiedLogger
from src.datasets.imagenet100 import build_dataloader, build_mae_transform, build_supervised_transform
from src.models.mae import build_mae
from src.models.vit_classifier import build_vit_classifier
from src.training.optimizer import build_optimizer, build_scheduler
from src.training.trainer_mae import MAETrainer
from src.training.trainer_supervised import SupervisedTrainer

set_seed(42)
device = get_device()

# ── Config smoke MAE: 2 epoche, batch ridotto, W&B abilitato ──────────────────
smoke_mae_cfg = copy.deepcopy(mae_cfg)
smoke_mae_cfg['training']['epochs'] = 2
smoke_mae_cfg['training']['warmup_epochs'] = 0
smoke_mae_cfg['training']['scheduler']['warmup_epochs'] = 0
smoke_mae_cfg['data']['batch_size'] = 32
smoke_mae_cfg['logging']['use_wandb'] = True   # verifica W&B
smoke_mae_cfg['experiment']['name'] = 'smoke_mae'
smoke_mae_cfg['checkpointing']['save_every'] = 999
smoke_mae_cfg['logging']['log_freq'] = 10

train_loader_mae = build_dataloader(
    root=DATA_ROOT, split='train',
    transform=build_mae_transform(224),
    batch_size=32, num_workers=2, drop_last=True,
)

model_mae = build_mae(smoke_mae_cfg).to(device)
print(f'MAE params: {count_parameters(model_mae):,}')

opt   = build_optimizer(model_mae, smoke_mae_cfg['training']['optimizer'])
sched = build_scheduler(opt, smoke_mae_cfg['training']['scheduler'], 2)
logger_smoke = UnifiedLogger(
    log_dir=str(EXP_DIR / 'logs'),
    experiment_name='smoke_mae',
    config=smoke_mae_cfg,
    use_wandb=True,
    use_tensorboard=False,
)

trainer_mae = MAETrainer(
    model=model_mae, optimizer=opt, scheduler=sched,
    train_loader=train_loader_mae, logger=logger_smoke,
    cfg=smoke_mae_cfg, device=device,
    checkpoint_dir=EXP_DIR / 'checkpoints' / 'smoke_mae',
)
trainer_mae.train()
logger_smoke.finish()
print('\n✓ Smoke test MAE completato — controlla W&B per verificare il logging.')

In [ ]:
# ── Config smoke Supervised: 2 epoche, batch ridotto, W&B abilitato ───────────
smoke_sup_cfg = copy.deepcopy(sup_cfg)
smoke_sup_cfg['training']['epochs'] = 2
smoke_sup_cfg['training']['warmup_epochs'] = 0
smoke_sup_cfg['training']['scheduler']['warmup_epochs'] = 0
smoke_sup_cfg['data']['batch_size'] = 32
smoke_sup_cfg['logging']['use_wandb'] = True   # verifica W&B
smoke_sup_cfg['experiment']['name'] = 'smoke_supervised'
smoke_sup_cfg['checkpointing']['save_every'] = 999
smoke_sup_cfg['logging']['log_freq'] = 10

train_loader_sup = build_dataloader(
    root=DATA_ROOT, split='train',
    transform=build_supervised_transform(224, is_train=True),
    batch_size=32, num_workers=2, drop_last=True,
)
val_loader = build_dataloader(
    root=DATA_ROOT, split='val',
    transform=build_supervised_transform(224, is_train=False),
    batch_size=64, num_workers=2,
)

model_sup = build_vit_classifier(smoke_sup_cfg, pretrained_encoder_path=None).to(device)
opt2   = build_optimizer(model_sup, smoke_sup_cfg['training']['optimizer'])
sched2 = build_scheduler(opt2, smoke_sup_cfg['training']['scheduler'], 2)
logger2 = UnifiedLogger(
    log_dir=str(EXP_DIR / 'logs'),
    experiment_name='smoke_supervised',
    config=smoke_sup_cfg,
    use_wandb=True,
    use_tensorboard=False,
)

trainer_sup = SupervisedTrainer(
    model=model_sup, optimizer=opt2, scheduler=sched2,
    train_loader=train_loader_sup, val_loader=val_loader,
    logger=logger2, cfg=smoke_sup_cfg, device=device,
    checkpoint_dir=EXP_DIR / 'checkpoints' / 'smoke_sup',
)
trainer_sup.train()
logger2.finish()
print('\n✓ Smoke test Supervised completato — controlla W&B per verificare il logging.')

## 4. MAE Pre-training (200 epoche)
Esegui solo dopo che i smoke test hanno passato. Kaggle concede max 12h per sessione GPU.

In [ ]:
from src.utils.checkpoint import load_checkpoint

set_seed(mae_cfg['experiment']['seed'])

train_loader_mae_full = build_dataloader(
    root=DATA_ROOT, split='train',
    transform=build_mae_transform(224),
    batch_size=mae_cfg['data']['batch_size'],
    num_workers=mae_cfg['data']['num_workers'],
    drop_last=True,
)

model_mae_full = build_mae(mae_cfg).to(device)
print(f'MAE params: {count_parameters(model_mae_full):,}')

opt_mae   = build_optimizer(model_mae_full, mae_cfg['training']['optimizer'])
sched_mae = build_scheduler(opt_mae, mae_cfg['training']['scheduler'], mae_cfg['training']['epochs'])

logger_mae = UnifiedLogger(
    log_dir=mae_cfg['logging']['log_dir'],
    experiment_name=mae_cfg['experiment']['name'],
    config=mae_cfg,
    use_wandb=mae_cfg['logging']['use_wandb'],
    use_tensorboard=mae_cfg['logging']['use_tensorboard'],
)

# Resume automatico se esiste un checkpoint
start_epoch = 0
ckpt_dir = Path(mae_cfg['checkpointing']['checkpoint_dir'])
existing = sorted(ckpt_dir.glob('checkpoint_epoch_*.pth'))
if existing:
    last_ckpt = existing[-1]
    print(f'Riprendendo da {last_ckpt.name} ...')
    ckpt = load_checkpoint(last_ckpt, model_mae_full, opt_mae, sched_mae, device)
    start_epoch = ckpt.get('epoch', 0) + 1

trainer_mae_full = MAETrainer(
    model=model_mae_full, optimizer=opt_mae, scheduler=sched_mae,
    train_loader=train_loader_mae_full, logger=logger_mae,
    cfg=mae_cfg, device=device,
    checkpoint_dir=ckpt_dir,
)
trainer_mae_full.global_step = start_epoch * len(train_loader_mae_full)
trainer_mae_full.train(start_epoch=start_epoch)
logger_mae.finish()

## 5. Supervised ViT Baseline (200 epoche)

In [ ]:
set_seed(sup_cfg['experiment']['seed'])

train_loader_sup_full = build_dataloader(
    root=DATA_ROOT, split='train',
    transform=build_supervised_transform(224, is_train=True),
    batch_size=sup_cfg['data']['batch_size'],
    num_workers=sup_cfg['data']['num_workers'],
    drop_last=True,
)
val_loader_full = build_dataloader(
    root=DATA_ROOT, split='val',
    transform=build_supervised_transform(224, is_train=False),
    batch_size=sup_cfg['data']['batch_size'],
    num_workers=sup_cfg['data']['num_workers'],
)

model_sup_full = build_vit_classifier(sup_cfg, pretrained_encoder_path=None).to(device)
print(f'ViTClassifier params: {count_parameters(model_sup_full):,}')

opt_sup   = build_optimizer(model_sup_full, sup_cfg['training']['optimizer'])
sched_sup = build_scheduler(opt_sup, sup_cfg['training']['scheduler'], sup_cfg['training']['epochs'])

logger_sup = UnifiedLogger(
    log_dir=sup_cfg['logging']['log_dir'],
    experiment_name=sup_cfg['experiment']['name'],
    config=sup_cfg,
    use_wandb=sup_cfg['logging']['use_wandb'],
    use_tensorboard=sup_cfg['logging']['use_tensorboard'],
)

start_epoch = 0
ckpt_dir_sup = Path(sup_cfg['checkpointing']['checkpoint_dir'])
existing_sup = sorted(ckpt_dir_sup.glob('checkpoint_epoch_*.pth'))
if existing_sup:
    last = existing_sup[-1]
    print(f'Riprendendo da {last.name} ...')
    ckpt = load_checkpoint(last, model_sup_full, opt_sup, sched_sup, device)
    start_epoch = ckpt.get('epoch', 0) + 1

trainer_sup_full = SupervisedTrainer(
    model=model_sup_full, optimizer=opt_sup, scheduler=sched_sup,
    train_loader=train_loader_sup_full, val_loader=val_loader_full,
    logger=logger_sup, cfg=sup_cfg, device=device,
    checkpoint_dir=ckpt_dir_sup,
)
trainer_sup_full.global_step = start_epoch * len(train_loader_sup_full)
trainer_sup_full.train(start_epoch=start_epoch)
logger_sup.finish()

## 6. Linear Probe (90 epoche)
Richiede che il MAE pre-training sia completato.

In [ ]:
from src.training.trainer_linear_probe import LinearProbeTrainer

mae_best_ckpt = Path(probe_cfg['model']['pretrained_encoder'])
if not mae_best_ckpt.exists():
    raise FileNotFoundError(
        f'MAE checkpoint non trovato: {mae_best_ckpt}\n'
        'Assicurati che il MAE pre-training (sezione 4) sia completato.'
    )

set_seed(probe_cfg['experiment']['seed'])

model_probe = build_vit_classifier(
    probe_cfg,
    pretrained_encoder_path=str(mae_best_ckpt),
    device=device,
).to(device)

trainable = count_parameters(model_probe)
total     = sum(p.numel() for p in model_probe.parameters())
print(f'Trainable: {trainable:,} / {total:,} (encoder frozen)')

opt_probe   = build_optimizer(model_probe, probe_cfg['training']['optimizer'])
sched_probe = build_scheduler(opt_probe, probe_cfg['training']['scheduler'],
                               probe_cfg['training']['epochs'])

logger_probe = UnifiedLogger(
    log_dir=probe_cfg['logging']['log_dir'],
    experiment_name=probe_cfg['experiment']['name'],
    config=probe_cfg,
    use_wandb=probe_cfg['logging']['use_wandb'],
    use_tensorboard=probe_cfg['logging']['use_tensorboard'],
)

# Riutilizza i dataloader della sezione 5 se già creati, altrimenti ricreali
if 'train_loader_sup_full' not in dir():
    train_loader_sup_full = build_dataloader(
        root=DATA_ROOT, split='train',
        transform=build_supervised_transform(224, is_train=True),
        batch_size=probe_cfg['data']['batch_size'],
        num_workers=probe_cfg['data']['num_workers'], drop_last=True,
    )
    val_loader_full = build_dataloader(
        root=DATA_ROOT, split='val',
        transform=build_supervised_transform(224, is_train=False),
        batch_size=probe_cfg['data']['batch_size'],
        num_workers=probe_cfg['data']['num_workers'],
    )

trainer_probe = LinearProbeTrainer(
    model=model_probe, optimizer=opt_probe, scheduler=sched_probe,
    train_loader=train_loader_sup_full, val_loader=val_loader_full,
    logger=logger_probe, cfg=probe_cfg, device=device,
    checkpoint_dir=Path(probe_cfg['checkpointing']['checkpoint_dir']),
)
trainer_probe.train()
logger_probe.finish()

## 7. Valutazione comparativa

In [ ]:
from src.evaluation.evaluator import Evaluator, print_comparison
from src.utils.checkpoint import load_checkpoint

if 'val_loader_full' not in dir():
    val_loader_full = build_dataloader(
        root=DATA_ROOT, split='val',
        transform=build_supervised_transform(224, is_train=False),
        batch_size=128, num_workers=4,
    )

results_all = []

for label, cfg, ckpt_name in [
    ('Supervised ViT (scratch)',  sup_cfg,   'checkpoint_best.pth'),
    ('MAE Encoder + Linear Probe', probe_cfg, 'checkpoint_best.pth'),
]:
    ckpt_path = Path(cfg['checkpointing']['checkpoint_dir']) / ckpt_name
    if not ckpt_path.exists():
        print(f'[SKIP] {label}: checkpoint non trovato ({ckpt_path})')
        continue
    model_eval = build_vit_classifier(cfg, pretrained_encoder_path=None).to(device)
    load_checkpoint(ckpt_path, model_eval, device=device)
    ev = Evaluator(model_eval, val_loader_full, device)
    res = ev.evaluate()
    results_all.append((label, res))
    ev.print_report(res, label=label)

if len(results_all) > 1:
    print_comparison(results_all)